# 03 – Modeling

**Project:** Diabetes Prediction ML
**Seminar:** Advanced Applied Data Science – Goethe Universität Frankfurt
**Dataset:** CDC BRFSS 2015 – Diabetes Health Indicators (UCI #891)
**CRISP-DM Phase:** 4 – Modeling

---

## Evaluation Strategy

### Why PR-AUC as Primary Metric?

With a class imbalance of ~86:14, standard metrics are misleading:

- **Accuracy** — a model predicting "No Diabetes" for every sample achieves ~86% accuracy. This metric is **reported but not optimised for**.
- **ROC-AUC** — measures rank ordering across all thresholds. It is less sensitive to class imbalance because it includes the large True Negative population, which inflates performance estimates.
- **PR-AUC (Average Precision)** — measures precision across all recall levels, focusing exclusively on the **minority class (Prediabetes/Diabetes)**. A no-skill classifier scores PR-AUC ≈ class prevalence (~0.14). Any meaningful model must exceed this baseline substantially.

**Primary metric: PR-AUC** — reported as the main comparison criterion across all models.
Secondary metrics reported: ROC-AUC, Recall (minority), F1 (minority), Accuracy.

### Cross-Validation Strategy

**Stratified 5-Fold CV** with `random_state=42` ensures:
1. Each fold preserves the original 86:14 class ratio
2. Results are robust against a single train/test split
3. Variance of metrics is quantified (mean ± std)
4. SMOTE (when used) is applied **inside** each fold to the training split only — preventing data leakage from synthetic samples into the validation set

## Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, accuracy_score,
    classification_report, confusion_matrix, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directories
PLOTS_DIR = os.path.join('..', 'results', 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

# Plot style – consistent with 01_data_exploration.ipynb
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

LABEL_MAP    = {0: 'No Diabetes', 1: 'Prediabetes/Diabetes'}
CLASS_COLORS = {0: '#4CAF50', 1: '#E53935'}

print('Libraries loaded.  Seed =', SEED)

## 1 · Load Dataset

In [ ]:
dataset = fetch_ucirepo(id=891)
X = dataset.data.features.astype(int)
y = dataset.data.targets.squeeze().astype(int)

print(f'Features : {X.shape[0]:,} samples  x  {X.shape[1]} columns')
print(f'Target   : {y.name}')
print()
for k, v in y.value_counts().sort_index().items():
    print(f'  {LABEL_MAP[k]:28s}: {v:,}  ({v / len(y) * 100:.1f}%)')

baseline_rate = y.mean()
print(f'\nNo-skill PR-AUC baseline: {baseline_rate:.4f}')

## 2 · Cross-Validation Framework

We define the CV splitter and a pair of helper functions that are reused for every model:
- `run_cv` — runs stratified K-Fold, collects per-fold predictions and metrics
- `print_summary` — aggregates and prints CV metrics with mean ± std

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'Stratified {N_SPLITS}-Fold CV  |  shuffle=True  |  seed={SEED}')


def evaluate_fold(y_true, y_pred, y_prob):
    '''Compute all metrics for a single fold.'''
    return {
        'pr_auc'   : average_precision_score(y_true, y_prob),
        'roc_auc'  : roc_auc_score(y_true, y_prob),
        'accuracy' : accuracy_score(y_true, y_pred),
        'report'   : classification_report(
            y_true, y_pred,
            target_names=list(LABEL_MAP.values()),
            output_dict=True
        ),
        'confusion' : confusion_matrix(y_true, y_pred),
        'y_true'    : np.array(y_true),
        'y_prob'    : np.array(y_prob),
    }


def run_cv(estimator, X, y, cv, label='Model'):
    '''Train/evaluate with stratified K-Fold; return per-fold result dicts.'''
    print(f'\n[{label}]')
    fold_results = []
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_tr,  X_val  = X.iloc[tr_idx],  X.iloc[val_idx]
        y_tr,  y_val  = y.iloc[tr_idx],  y.iloc[val_idx]
        estimator.fit(X_tr, y_tr)
        y_pred = estimator.predict(X_val)
        y_prob = estimator.predict_proba(X_val)[:, 1]
        res = evaluate_fold(y_val, y_pred, y_prob)
        res['fold'] = fold
        fold_results.append(res)
        print(f'  Fold {fold}:  PR-AUC={res["pr_auc"]:.4f}  '
              f'ROC-AUC={res["roc_auc"]:.4f}  '
              f'Acc={res["accuracy"]:.4f}')
    return fold_results


def print_summary(results, label):
    '''Print mean +/- std for all metrics and the aggregated classification report.'''
    pr  = [r['pr_auc']   for r in results]
    roc = [r['roc_auc']  for r in results]
    acc = [r['accuracy'] for r in results]

    print(f'\n=== {label} — {N_SPLITS}-Fold Summary ===')
    print(f'  PR-AUC  (primary)   : {np.mean(pr):.4f}  +/-  {np.std(pr):.4f}')
    print(f'  ROC-AUC             : {np.mean(roc):.4f}  +/-  {np.std(roc):.4f}')
    print(f'  Accuracy (biased!)  : {np.mean(acc):.4f}  +/-  {np.std(acc):.4f}')
    print(f'  NOTE: Accuracy is {np.mean(acc)*100:.1f}% but is inflated by the 86% majority class.')

    classes = list(LABEL_MAP.values()) + ['macro avg', 'weighted avg']
    rows = {}
    for cls in classes:
        if cls in results[0]['report'] and isinstance(results[0]['report'][cls], dict):
            rows[cls] = {
                m: np.mean([r['report'][cls][m] for r in results])
                for m in ['precision', 'recall', 'f1-score', 'support']
            }
    report_df = pd.DataFrame(rows).T
    report_df['support'] = report_df['support'].astype(int)
    print('\n  Classification Report (mean across folds):')
    print(report_df.round(4).to_string())

    return np.mean(pr), np.mean(roc), np.mean(acc)

## 3 · Baseline 1 — Logistic Regression + class_weight='balanced'

**Strategy:** The `class_weight='balanced'` parameter scales the loss function so that the minority class contributes proportionally more to the gradient. Concretely, each minority sample receives a weight of `n_samples / (n_classes * n_minority_samples)` ≈ 3.6×.

**Advantage:** No synthetic data generation — the original data distribution is preserved.
**Disadvantage:** Linear decision boundary; the model may not learn non-linear patterns in the data.

In [ ]:
lr_cw = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=SEED
)

results_cw = run_cv(lr_cw, X, y, skf, label='LR + class_weight="balanced"')
pr_cw, roc_cw, acc_cw = print_summary(results_cw, 'LR + class_weight="balanced"')

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = fig.add_gridspec(1, 2, wspace=0.35)

# ── Confusion Matrix (mean across folds) ──────────────────────────────────────
ax_cm = fig.add_subplot(gs[0])
avg_cm = np.round(np.mean([r['confusion'] for r in results_cw], axis=0)).astype(int)
sns.heatmap(avg_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(LABEL_MAP.values()),
            yticklabels=list(LABEL_MAP.values()),
            ax=ax_cm, cbar=False, linewidths=0.5)
ax_cm.set_xlabel('Predicted')
ax_cm.set_ylabel('Actual')
ax_cm.set_title('Confusion Matrix\n(mean across 5 folds)', fontweight='bold')

# ── PR Curves (one per fold) ──────────────────────────────────────────────────
ax_pr = fig.add_subplot(gs[1])
for r in results_cw:
    prec, rec, _ = precision_recall_curve(r['y_true'], r['y_prob'])
    ax_pr.plot(rec, prec, alpha=0.55, lw=1.3, color='#4E79A7')
ax_pr.axhline(baseline_rate, color='gray', linestyle='--', lw=1.2,
              label=f'No-skill baseline ({baseline_rate:.2f})')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_xlim([0, 1])
ax_pr.set_ylim([0, 1.05])
ax_pr.set_title(f'PR Curves — {N_SPLITS} Folds\nMean PR-AUC = {pr_cw:.4f}',
                fontweight='bold')
ax_pr.legend(fontsize=9)

fig.suptitle('Baseline 1: LR + class_weight="balanced"',
             fontsize=13, fontweight='bold', y=1.02)
plt.savefig(os.path.join(PLOTS_DIR, '15_lr_classweight_results.png'), bbox_inches='tight')
plt.show()

**Interpretation:**
The confusion matrix reveals the trade-off of `class_weight='balanced'`: the model improves **recall** for the minority class (Prediabetes/Diabetes) compared to an unweighted baseline, but at the cost of increased false positives (predicting diabetes for healthy individuals). In a public health screening context, **higher recall is preferred** — missing a true diabetic (false negative) is more harmful than a false alarm that prompts further testing.

The PR curves across all 5 folds are closely clustered, indicating stable generalisation. The mean PR-AUC meaningfully exceeds the no-skill baseline (~0.14).

## 4 · Baseline 2 — Logistic Regression + SMOTE

**Strategy:** SMOTE (Synthetic Minority Oversampling Technique) generates synthetic minority-class samples by interpolating between existing minority samples in feature space. The training set is balanced to 50:50 before fitting the logistic regression.

**Critical implementation note:** SMOTE is placed **inside** an `imblearn.Pipeline`. This guarantees that:
- Synthetic samples are generated **only from the training fold**
- The validation fold contains only real, original samples
- No information from validation samples is used to generate synthetic training data

This is the correct, leak-free implementation. Applying SMOTE before the CV split would constitute data leakage and produce optimistically biased results.

In [ ]:
lr_smote = ImbPipeline([
    ('smote', SMOTE(random_state=SEED)),
    ('lr',    LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED))
])

results_smote = run_cv(lr_smote, X, y, skf, label='LR + SMOTE')
pr_smote, roc_smote, acc_smote = print_summary(results_smote, 'LR + SMOTE')

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = fig.add_gridspec(1, 2, wspace=0.35)

ax_cm = fig.add_subplot(gs[0])
avg_cm_s = np.round(np.mean([r['confusion'] for r in results_smote], axis=0)).astype(int)
sns.heatmap(avg_cm_s, annot=True, fmt='d', cmap='Oranges',
            xticklabels=list(LABEL_MAP.values()),
            yticklabels=list(LABEL_MAP.values()),
            ax=ax_cm, cbar=False, linewidths=0.5)
ax_cm.set_xlabel('Predicted')
ax_cm.set_ylabel('Actual')
ax_cm.set_title('Confusion Matrix\n(mean across 5 folds)', fontweight='bold')

ax_pr = fig.add_subplot(gs[1])
for r in results_smote:
    prec, rec, _ = precision_recall_curve(r['y_true'], r['y_prob'])
    ax_pr.plot(rec, prec, alpha=0.55, lw=1.3, color='#F28E2B')
ax_pr.axhline(baseline_rate, color='gray', linestyle='--', lw=1.2,
              label=f'No-skill baseline ({baseline_rate:.2f})')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_xlim([0, 1])
ax_pr.set_ylim([0, 1.05])
ax_pr.set_title(f'PR Curves — {N_SPLITS} Folds\nMean PR-AUC = {pr_smote:.4f}',
                fontweight='bold')
ax_pr.legend(fontsize=9)

fig.suptitle('Baseline 2: LR + SMOTE',
             fontsize=13, fontweight='bold', y=1.02)
plt.savefig(os.path.join(PLOTS_DIR, '16_lr_smote_results.png'), bbox_inches='tight')
plt.show()

**Interpretation:**
SMOTE creates synthetic minority-class samples in feature space, effectively teaching the model more decision boundary nuance around the minority class. Compared to `class_weight='balanced'`, SMOTE typically achieves higher recall at the cost of a different precision–recall trade-off.

Note that SMOTE increases training set size (from ~203k to ~346k training samples per fold due to oversampling to 50:50), which increases training time. For more complex models (Random Forest, XGBoost) this runtime cost becomes relevant.

## 5 · Baseline Comparison

Both approaches represent the weakest reasonable baseline for this problem — they establish the floor that more complex models must beat. The comparison table and bar chart summarise performance across all reported metrics.

In [ ]:
comparison = pd.DataFrame({
    'LR + class_weight': {
        'PR-AUC (primary)': np.mean([r['pr_auc']  for r in results_cw]),
        'ROC-AUC'         : np.mean([r['roc_auc'] for r in results_cw]),
        'Accuracy'        : np.mean([r['accuracy'] for r in results_cw]),
        'Recall (pos.)'   : np.mean([r['report']['Prediabetes/Diabetes']['recall']
                                     for r in results_cw]),
        'F1 (pos.)'       : np.mean([r['report']['Prediabetes/Diabetes']['f1-score']
                                     for r in results_cw]),
    },
    'LR + SMOTE': {
        'PR-AUC (primary)': np.mean([r['pr_auc']  for r in results_smote]),
        'ROC-AUC'         : np.mean([r['roc_auc'] for r in results_smote]),
        'Accuracy'        : np.mean([r['accuracy'] for r in results_smote]),
        'Recall (pos.)'   : np.mean([r['report']['Prediabetes/Diabetes']['recall']
                                     for r in results_smote]),
        'F1 (pos.)'       : np.mean([r['report']['Prediabetes/Diabetes']['f1-score']
                                     for r in results_smote]),
    }
}).round(4)

print('=== Baseline Comparison (mean across 5 folds) ===')
print(comparison.T.to_string())

# ── Bar chart comparison ───────────────────────────────────────────────────────
metrics_plot = ['PR-AUC (primary)', 'ROC-AUC', 'Recall (pos.)', 'F1 (pos.)']
x = np.arange(len(metrics_plot))
width = 0.32

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width / 2,
               [comparison.loc[m, 'LR + class_weight'] for m in metrics_plot],
               width, label='LR + class_weight', color='#4E79A7', alpha=0.88,
               edgecolor='white')
bars2 = ax.bar(x + width / 2,
               [comparison.loc[m, 'LR + SMOTE'] for m in metrics_plot],
               width, label='LR + SMOTE', color='#F28E2B', alpha=0.88,
               edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.006,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_plot, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Logistic Regression Baselines — Metric Comparison\n'
             '(Stratified 5-Fold CV, mean across folds)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(baseline_rate, color='gray', linestyle=':', lw=1.2,
           label=f'No-skill PR-AUC ({baseline_rate:.2f})')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '17_baseline_comparison.png'), bbox_inches='tight')
plt.show()

## 6 · Ergebnisse und Interpretation der Baseline-Modelle

### Evaluierungsrahmen

Die Evaluation beider Baseline-Modelle erfolgte mit Stratified 5-Fold Cross-Validation (Seed=42). Als primäre Metrik wurde PR-AUC (Average Precision) gewählt, da das Dataset mit einer Klassenverteilung von 86:14 stark imbalanciert ist. Ein naives Modell, das ausnahmslos "kein Diabetes" vorhersagt, erreicht zwar ~86% Accuracy, aber einen PR-AUC von nur 0.1393 — entsprechend der bloßen Prävalenz der Minderheitsklasse. Jedes sinnvolle Modell muss diesen Wert substanziell übertreffen.

### Ergebnisse im Überblick

**LR + class_weight='balanced'** erzielte einen mittleren PR-AUC von **0.4031 ± 0.0021** über alle fünf Folds. Das entspricht einer knapp dreifachen Verbesserung gegenüber der No-skill Baseline (0.1393). Der ROC-AUC liegt bei **0.8225 ± 0.0033**, was auf eine gute Trennfähigkeit zwischen den Klassen hinweist. Die Accuracy beträgt **73.2%** — sie ist jedoch mit Vorsicht zu interpretieren, da das Modell durch die erhöhten Klassengewichte gezielt mehr Minderheitsklassen-Fehler in Kauf nimmt und entsprechend Mehrheitsklassen-Genauigkeit opfert.

**LR + SMOTE** schneidet auf allen Metriken schlechter ab: PR-AUC **0.3441 ± 0.0047**, ROC-AUC **0.7808 ± 0.0030**, Recall **0.6880 ± 0.0064**. Der Rückstand gegenüber dem class_weight-Ansatz ist auf allen Ebenen konsistent und statistisch stabil.

### Warum class_weight SMOTE schlägt

Dieses Ergebnis ist auf den ersten Blick überraschend, lässt sich aber durch die Natur des Datasets erklären. SMOTE erzeugt synthetische Minority-Samples durch lineare Interpolation zwischen existierenden Samples im Feature-Raum. Bei einem Dataset mit 14 binären Features (Werte nur 0 oder 1) und 4 ordinalen Features entstehen dabei Zwischenwerte wie `HighBP = 0.3` oder `Smoker = 0.7`, die keine realen Beobachtungen repräsentieren und klinisch bedeutungslos sind. Das Modell lernt dadurch auf Basis unrealistischer Datenpunkte, was die Generalisierung auf echte Validierungsdaten verschlechtert. Der class_weight-Ansatz hingegen verändert die Daten überhaupt nicht — er skaliert lediglich die Verlustfunktion so, dass Fehler bei der Minderheitsklasse stärker bestraft werden. Für dieses Survey-basierte, kategorial-dominierte Dataset ist das die robustere Strategie.

### Confusion Matrix und klinische Relevanz

Im Durchschnitt über alle fünf Folds identifiziert das beste Modell (LR + class_weight) pro Fold **5.411 Diabetiker korrekt (True Positives)**, übersieht aber **1.658 Diabetiker (False Negatives)** und löst **11.962 Fehlalarme (False Positives)** aus. Das entspricht einem Recall von **76.5%**: das Modell erkennt etwa drei von vier Diabetikern.

Im klinischen Screening-Kontext ist der Recall die entscheidende Größe. Ein verpasster Diabetiker (False Negative) bedeutet ausbleibende Behandlung und das Risiko von Folgeerkrankungen — das Paper von Burrows et al. (2017) dokumentiert genau diese Progressionskette von unbehandeltem Diabetes zu Nierenversagen. Ein Fehlalarm (False Positive) hingegen führt lediglich zu einem weiteren klinischen Test. Trotzdem zeigt der Recall von 76.5%, dass 23.5% der Erkrankten noch immer vom Modell übersehen werden — eine Lücke, die komplexere, nicht-lineare Modelle in den nächsten Schritten schließen sollen.

### Fazit und Performance Floor

Beide Logistic-Regression-Varianten bestätigen, dass das Problem mit linearen Methoden prinzipiell lösbar ist — der PR-AUC von 0.40 liegt weit über dem Zufallsniveau. Gleichzeitig macht die vergleichsweise niedrige Precision von 0.31 deutlich, dass Logistische Regression die komplexen Interaktionen zwischen den Features (z. B. das Zusammenspiel von Alter, BMI und Blutdruck) nicht vollständig abbilden kann.

Für alle weiteren Modelle gilt: PR-AUC > **0.40** und Recall > **0.77** sind die Mindestanforderungen für eine echte Verbesserung gegenüber dieser Baseline. Ein Modell, das diese Schwelle nicht überschreitet, bietet keinen Mehrwert gegenüber dem einfachsten sinnvollen Ansatz.